In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = '/content/drive/MyDrive/GAMEEMO_processed/gameemo_preprocessed.pkl'
SAVE_DIR = '/content/drive/MyDrive/GAMEEMO_processed/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
subject_ids = data['subject_ids']
n_classes = len(data['label_names'])

trial_ids = np.array([f'{s}_{l}' for s, l in zip(subject_ids, y)])

print('X shape:', X.shape)
print('classes:', data['label_names'])
print('total trials:', len(np.unique(trial_ids)))

X shape: (8288, 512, 14)
classes: ['boring', 'calm', 'horror', 'funny']
total trials: 112


In [ ]:
def get_subject_dependent_split(X, y, trial_ids, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=trial_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def get_cross_subject_split(X, y, subject_ids, test_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=subject_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train_sd, X_test_sd, y_train_sd, y_test_sd = get_subject_dependent_split(X, y, trial_ids)
X_train_cs, X_test_cs, y_train_cs, y_test_cs = get_cross_subject_split(X, y, subject_ids)

print('subject dependent train/test:', X_train_sd.shape, X_test_sd.shape)
print('cross subject train/test:', X_train_cs.shape, X_test_cs.shape)

subject dependent train/test: (6586, 512, 14) (1702, 512, 14)
cross subject train/test: (6512, 512, 14) (1776, 512, 14)


In [ ]:
def build_cnn_lstm(n_timepoints, n_channels, n_classes):
    inputs = layers.Input(shape=(n_timepoints, n_channels))

    x = layers.Conv1D(32, kernel_size=7, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
def build_cnn_bigru(n_timepoints, n_channels, n_classes):
    inputs = layers.Input(shape=(n_timepoints, n_channels))

    x = layers.Conv1D(32, kernel_size=7, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Bidirectional(layers.GRU(64, return_sequences=False))(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name, protocol_name, epochs=50, batch_size=32):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    checkpoint_path = os.path.join(SAVE_DIR, f'{model_name}_{protocol_name}_best.keras')

    callback_list = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callback_list,
        verbose=1
    )

    y_pred = np.argmax(model.predict(X_test), axis=1)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f'{model_name} ({protocol_name}) accuracy: {acc:.4f}, f1: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=data['label_names']))

    result = {
        'model': model_name,
        'protocol': protocol_name,
        'accuracy': acc,
        'f1_score': f1,
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'history': history.history
    }

    return result, model

# CNN-LSTM subject dependent

In [ ]:
cnn_lstm_model = build_cnn_lstm(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_lstm_sd, _ = train_and_evaluate(
    cnn_lstm_model, X_train_sd, y_train_sd, X_test_sd, y_test_sd,
    model_name='cnn_lstm', protocol_name='subject_dependent'
)

Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - accuracy: 0.3294 - loss: 1.3565 - val_accuracy: 0.3310 - val_loss: 1.3595 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.4773 - loss: 1.1632 - val_accuracy: 0.3289 - val_loss: 1.3886 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5997 - loss: 0.9731 - val_accuracy: 0.4018 - val_loss: 1.4656 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.6106 - loss: 0.9749 - val_accuracy: 0.4200 - val_loss: 1.4474 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.6799 - loss: 0.8266 - val_accuracy: 0.4302 - val_loss: 1.4187 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6885 - loss: 0.7998 - val_accuracy: 0.4545 - val_loss: 1.5278 - learning_rate: 0.0010
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.7435 - loss: 0

# CNN-LSTM Cross Subject

In [ ]:
cnn_lstm_model_2 = build_cnn_lstm(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_lstm_cs, _ = train_and_evaluate(
    cnn_lstm_model_2, X_train_cs, y_train_cs, X_test_cs, y_test_cs,
    model_name='cnn_lstm', protocol_name='cross_subject'
)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.3344 - loss: 1.3595 - val_accuracy: 0.3234 - val_loss: 1.3846 - learning_rate: 0.0010
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.4482 - loss: 1.2106 - val_accuracy: 0.3797 - val_loss: 1.2711 - learning_rate: 0.0010
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.5023 - loss: 1.1476 - val_accuracy: 0.4964 - val_loss: 1.1926 - learning_rate: 0.0010
Epoch 4/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.5727 - loss: 1.0238 - val_accuracy: 0.5005 - val_loss: 1.1787 - learning_rate: 0.0010
Epoch 5/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6508 - loss: 0.8799 - val_accuracy: 0.5834 - val_loss: 0.9796 - learning_rate: 0.0010
Epoch 6/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.6734 - loss: 0.8085 - val_accuracy: 0.6479 - val_loss: 0.7897 - learning_rate: 0.0010
Epoch 7/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.6925 - loss: 0.

# CNN-BiGRU Subject Dependent

In [ ]:
cnn_bigru_model = build_cnn_bigru(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_bigru_sd, _ = train_and_evaluate(
    cnn_bigru_model, X_train_sd, y_train_sd, X_test_sd, y_test_sd,
    model_name='cnn_bigru', protocol_name='subject_dependent'
)

Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 7s 20ms/step - accuracy: 0.2856 - loss: 1.4193 - val_accuracy: 0.2642 - val_loss: 1.3923 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.3776 - loss: 1.2964 - val_accuracy: 0.3654 - val_loss: 1.3098 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.5139 - loss: 1.0990 - val_accuracy: 0.4453 - val_loss: 1.2032 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.6191 - loss: 0.9034 - val_accuracy: 0.4889 - val_loss: 1.2023 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.7099 - loss: 0.7229 - val_accuracy: 0.5364 - val_loss: 1.1152 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7717 - loss: 0.5922 - val_accuracy: 0.5597 - val_loss: 1.1144 - learning_rate: 0.0010
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - accuracy: 0.8160 - loss: 0.

# CNN-BiGRU Cross Subject

In [ ]:
cnn_bigru_model_2 = build_cnn_bigru(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_bigru_cs, _ = train_and_evaluate(
    cnn_bigru_model_2, X_train_cs, y_train_cs, X_test_cs, y_test_cs,
    model_name='cnn_bigru', protocol_name='cross_subject'
)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.2829 - loss: 1.4232 - val_accuracy: 0.3193 - val_loss: 1.3708 - learning_rate: 0.0010
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - accuracy: 0.3810 - loss: 1.3086 - val_accuracy: 0.3593 - val_loss: 1.3145 - learning_rate: 0.0010
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.4744 - loss: 1.1535 - val_accuracy: 0.4667 - val_loss: 1.1791 - learning_rate: 0.0010
Epoch 4/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.5803 - loss: 0.9820 - val_accuracy: 0.5445 - val_loss: 1.0796 - learning_rate: 0.0010
Epoch 5/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.6580 - loss: 0.8235 - val_accuracy: 0.6049 - val_loss: 0.9016 - learning_rate: 0.0010
Epoch 6/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 4s 24ms/step - accuracy: 0.7174 - loss: 0.7011 - val_accuracy: 0.6909 - val_loss: 0.7649 - learning_rate: 0.0010
Epoch 7/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 3s 17ms/step - accuracy: 0.7664 - loss: 0.

# Save Results

In [ ]:
day3_results = {
    'cnn_lstm_subject_dependent': result_cnn_lstm_sd,
    'cnn_lstm_cross_subject': result_cnn_lstm_cs,
    'cnn_bigru_subject_dependent': result_cnn_bigru_sd,
    'cnn_bigru_cross_subject': result_cnn_bigru_cs,
}

results_path = os.path.join(SAVE_DIR, 'day3_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(day3_results, f)

print('saved day 3 results to', results_path)

for key, res in day3_results.items():
    print(key, '- accuracy:', round(res['accuracy'], 4), 'f1:', round(res['f1_score'], 4))

saved day 3 results to /content/drive/MyDrive/GAMEEMO_processed/results/day3_results.pkl
cnn_lstm_subject_dependent - accuracy: 0.6745 f1: 0.6869
cnn_lstm_cross_subject - accuracy: 0.9566 f1: 0.9568
cnn_bigru_subject_dependent - accuracy: 0.6845 f1: 0.6963
cnn_bigru_cross_subject - accuracy: 0.9702 f1: 0.9702
